In [1]:
import pandas as pd
import numpy as np
import os
import ast 
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

anova_kaggle = {}
anova_mendeley = {}

chi2_kaggle = {}
chi2_mendeley = {}

mi_kaggle = {}
mi_mendeley = {}

vt_kaggle = {}
vt_mendeley = {}
for file in (os.listdir('../results/phase_2')):
    if 'final_summary' in file:
        df = pd.read_csv(f'../results/phase_2/{file}')
        kaggle_features = df.loc[df['Dataset'] == 'Kaggle', ["Method", "Model", "Features_Used"]]
        for index, row in kaggle_features.iterrows():
    
            method = row["Method"]
            model_name = row["Model"]
            feature_list = ast.literal_eval(row["Features_Used"])
            
            match method:
                case "Anova":
                    anova_kaggle[model_name] = feature_list
                case "Variance Threshold":
                    vt_kaggle[model_name] = feature_list
                case "Mutual Info":
                    mi_kaggle[model_name] = feature_list
                case "Chi Square":
                    chi2_kaggle[model_name] = feature_list
        
        mendeley_features = df.loc[df['Dataset'] == 'Mendeley', ["Method", "Model", "Features_Used"]]
        for index, row in mendeley_features.iterrows():
            method = row["Method"]
            model_name = row["Model"]
            feature_list = ast.literal_eval(row["Features_Used"])
            
            match method:
                case "Anova":
                    anova_mendeley[model_name] = feature_list
                case "Variance Threshold":
                    vt_mendeley[model_name] = feature_list
                case "Mutual Info":
                    mi_mendeley[model_name] = feature_list
                case "Chi Square":
                    chi2_mendeley[model_name] = feature_list


In [2]:
from data_preprocessing import create_train_test_val_sets, get_processed_df

#Create test train splits
x_mendeley, y_mendeley = get_processed_df(r"..\data\raw\Mendeley Dataset.csv")
x_kaggle, y_kaggle= get_processed_df(r"..\data\raw\dataset_phishing.csv")

mendeley_sets = create_train_test_val_sets(x_mendeley,y_mendeley, label_col="Label", test_size=0.2, n_splits=5)
kaggle_sets = create_train_test_val_sets(x_kaggle,y_kaggle, label_col="Label", test_size=0.2, n_splits=5)

----------Processing None Dataset----------

Class Distribution:
Label
0    0.518415
1    0.481585
Name: proportion, dtype: float64
int64

Total Missing Values: 0
No categorical features to hash
Shape After Processing: (247950, 42)
True
----------Processing None Dataset----------

Class Distribution:
Label
legitimate    0.5
phishing      0.5
Name: proportion, dtype: float64
object

Total Missing Values: 0
Shape After Processing: (11430, 32856)
True
Train/validation/test split prepared: 210757 instances for training, 37193 instances for validation, 49590 instances for testing
Stratified 5-fold CV splits created.
Train/validation/test split prepared: 9715 instances for training, 1715 instances for validation, 2286 instances for testing
Stratified 5-fold CV splits created.


In [3]:
import joblib 

#import phase 1 models
xgb_mendeley = joblib.load('./models/phase_1/xgboost_mendeley_no_fs.joblib')
xgb_kaggle = joblib.load('./models/phase_1/xgboost_kaggle_no_fs.joblib')
logreg_mendeley = joblib.load('./models/phase_1/logreg_mendeley_no_fs.joblib')
logreg_kaggle = joblib.load('./models/phase_1/logreg_kaggle_no_fs.joblib')
rf_mendeley = joblib.load('./models/phase_1/rf_mendeley_no_fs.joblib')
rf_kaggle = joblib.load('./models/phase_1/rf_kaggle_no_fs.joblib')

In [4]:
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif

from sklearn.feature_selection import mutual_info_classif, chi2, f_classif
from sklearn.metrics.pairwise import cosine_similarity
import scipy.sparse as sp
import pandas as pd
import numpy as np

def score_feature_redundancy(dataset, dataset_name, model_name, method, selected_features):

    X_selected = dataset['x_train'][selected_features]
    y = dataset['y_train']

    #use chi2 because more optimized for sprase matrices
    if 'kaggle' in dataset_name.lower():      
        X_sparse = sp.csr_matrix(X_selected.values)
        
        #calculate relevance score 
        relevance_scores, _ = f_classif(X_sparse, y)
        relevance_scores = np.nan_to_num(relevance_scores, nan=0.0)
        
        #calculate redundancy using cosine similarity
        similarity_matrix = cosine_similarity(X_sparse.T)
        np.fill_diagonal(similarity_matrix, 0)
        max_redundancy_vals = similarity_matrix.max(axis=0)
        mean_redundancy_vals = similarity_matrix.mean(axis=0)
    else:        
        #calculate relevance score 
        relevance_scores = mutual_info_classif(X_selected, y, discrete_features='auto', random_state=42)
        
        #calculate redundancy using pearson correlation
        corr_matrix = X_selected.corr().abs()
        np.fill_diagonal(corr_matrix.values, 0)
        max_redundancy_vals = corr_matrix.max().values
        mean_redundancy_vals = corr_matrix.mean().values

    #normalize scores
    max_relevance = relevance_scores.max()
    relevance_normalized = relevance_scores / max_relevance if max_relevance > 0 else relevance_scores

    feature_scores = pd.DataFrame({
        'Feature': X_selected.columns,
        'Relevance_to_Target': relevance_normalized,
        'Max_Redundancy_Score': max_redundancy_vals,
        'Mean_Redundancy_Score': mean_redundancy_vals
    })
    feature_scores = feature_scores.sort_values(by='Relevance_to_Target', ascending=False).reset_index(drop=True)

    return feature_scores

def remove_redundant_features(dataset, dataset_name, model_name, method, selected_features, max_redundancy=0.85, min_relevance=0.01):
    print(f'Redundnacy Calculation for {dataset_name}')
    print(f'Initial feature counts {len(selected_features)}')

    scores_df = score_feature_redundancy(dataset, dataset_name, model_name, method, selected_features)
    
    #apply the filters based on the thresholds
    final_features_df = scores_df[
        (scores_df['Max_Redundancy_Score'] < max_redundancy) & 
        (scores_df['Relevance_to_Target'] > min_relevance)
    ]
    
    #extract the final list of feature names
    final_feature_list = final_features_df['Feature'].tolist()
    
    print(f"Removed {len(selected_features) - len(final_feature_list)} features.")
    print(f"Final feature count for {model_name}: {len(final_feature_list)}\n")
    
    return final_feature_list



In [5]:
#Evaluate and save 
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler
from sklearn.linear_model import LogisticRegression


def evaluate_model(model_name, dataset, model, selected_features):
    
    X_train = pd.concat([dataset["x_train"], dataset["x_val"]]).reindex(columns=selected_features)
    X_test = dataset["x_test"].reindex(columns=selected_features)
    y_train = pd.concat([dataset["y_train"], dataset["y_val"]])
    y_test = dataset["y_test"]

    if model_name == 'LogReg':
        params = model.named_steps["model"].get_params()
        print(params)
        model_clone = Pipeline([
            ("scaler", MaxAbsScaler()),
            ("model", LogisticRegression(**params))
        ])
    else:
        model_clone = clone(model)
    model_clone.fit(X_train, y_train)

    y_pred = model_clone.predict(X_test)

    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)

    return f1, prec, rec

In [6]:
#Run Mendeley
models_mendeley = {
    "XGBoost": xgb_mendeley,
    "RF": rf_mendeley,
    "LogReg": logreg_mendeley
}

selected_features = remove_redundant_features(
            mendeley_sets,
            "Mendeley",
            model_name,
            method,
            mendeley_sets['x_train'].columns
)

final_results_mendeley = []

for model_name, model in models_mendeley.items():
    print(f"\nEvaluating {model_name} - {method}")

    f1, prec, rec = evaluate_model(model_name, mendeley_sets, model, selected_features)

    print(f"{model_name} | F1: {f1:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")

    final_results_mendeley.append({
        "dataset": "Mendeley",
        "model": model_name,
        "method": method,
        "num_features": len(selected_features),
        "f1": f1,
        "precision": prec,
        "recall": rec,
        "features": selected_features
    })

os.makedirs(f"../results/phase_3/redundancy_reports", exist_ok=True)
final_results_mendeley_df = pd.DataFrame(final_results_mendeley)
final_results_mendeley_df.to_csv(f"../results/phase_3/redundancy_reports/mendeley_redundancy_scores.csv", index=False)

Redundnacy Calculation for Mendeley
Initial feature counts 41
Removed 21 features.
Final feature count for RF: 20


Evaluating XGBoost - Variance Threshold
XGBoost | F1: 0.9465, Precision: 0.9615, Recall: 0.9319

Evaluating RF - Variance Threshold
RF | F1: 0.9983, Precision: 0.9978, Recall: 0.9989

Evaluating LogReg - Variance Threshold
{'C': 10, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 5000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'saga', 'tol': 0.001, 'verbose': 0, 'warm_start': False}
LogReg | F1: 0.7601, Precision: 0.8030, Recall: 0.7215


In [7]:
#Run Kaggle
models_kaggle = {
    "XGBoost": xgb_kaggle,
    "RF": rf_kaggle,
    "LogReg": logreg_kaggle
}

selected_features = remove_redundant_features(
            kaggle_sets,
            "Kaggle",
            model_name,
            method,
            kaggle_sets['x_train'].columns,
            min_relevance=0.0
)

#Evaluate and save kaggle
final_results_kaggle = []

for model_name, model in models_kaggle.items():
    print(f"\nEvaluating {model_name} - {method}")
    f1, prec, rec = evaluate_model(model_name, kaggle_sets, model, selected_features)

    print(f"{model_name} | {method} - F1: {f1:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")

    final_results_kaggle.append({
        "dataset": "Kaggle",
        "model": model_name,
        "method": method,
        "num_features": len(selected_features),
        "f1": f1,
        "precision": prec,
        "recall": rec,
        "features": selected_features
    })

final_results_kaggle_df = pd.DataFrame(final_results_kaggle)
final_results_kaggle_df.to_csv(f"../results/phase_3/redundancy_reports/kaggle_redundancy_scores.csv", index=False)

Redundnacy Calculation for Kaggle
Initial feature counts 32855


C:\Users\nive0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


Removed 821 features.
Final feature count for LogReg: 32034


Evaluating XGBoost - Variance Threshold
XGBoost | Variance Threshold - F1: 1.0000, Precision: 1.0000, Recall: 1.0000

Evaluating RF - Variance Threshold
RF | Variance Threshold - F1: 1.0000, Precision: 1.0000, Recall: 1.0000

Evaluating LogReg - Variance Threshold
{'C': 10, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 5000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'saga', 'tol': 0.001, 'verbose': 0, 'warm_start': False}
LogReg | Variance Threshold - F1: 1.0000, Precision: 1.0000, Recall: 1.0000


In [ ]:
#XGBoost + Mendeley
def save_stability_redundancy(file_name, model_name, model, dataset_name, dataset, score_threshold=0.3):
    stability_df = pd.read_csv(file_name)
    stability_df["Stable features"] = stability_df["Stable features"].apply(ast.literal_eval)
    final_results = []

    for _, row in stability_df.iterrows():
        method = row["FS_Method"]
        stable_features_dict = row["Stable features"]

        print(f"\nEvaluating Composite Score for {model_name} - {method}")
        feature_names = list(stable_features_dict.keys())

        #get relevance and redundancy scores
        scores_df = score_feature_redundancy(dataset, dataset_name, model_name, method, feature_names)

        #add stability score
        scores_df['Stability_Score'] = scores_df['Feature'].map(stable_features_dict)

        #calculate (stability + relevance) - redundancy
        scores_df['Composite_Score'] = (scores_df['Stability_Score'] + scores_df['Relevance_to_Target']) - scores_df['Max_Redundancy_Score']

        #sort by the new composite score
        scores_df = scores_df.sort_values(by='Composite_Score', ascending=False).reset_index(drop=True)

        #filter for best features
        best_features_df = scores_df[
            (scores_df['Composite_Score'] > score_threshold) &
            (scores_df['Max_Redundancy_Score'] < 0.85) 
        ]

        final_features = best_features_df['Feature'].tolist()

        print(f"Selected {len(final_features)} out of {len(feature_names)} features.")

        # 5. Evaluate the model using only the best features
        f1, prec, rec = evaluate_model(model_name, dataset, model, final_features)

        final_results.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "FS_Method": method,
            "Stability_F1": row["Avg_Val_F1"],
            "Num_Stable_Features": len(stable_features_dict),
            "Num_Final_Features": len(final_features),
            "F1": f1,
            "Precision": prec,
            "Recall": rec,
            "Composite_Threshold": score_threshold,
            "Final_Features": final_features
        })

        # Optional: Save the detailed feature scoreboard for this specific run for your report
        os.makedirs(f"../results/phase_3/detailed_feature_scores", exist_ok=True)
        scores_df.to_csv(f"../results/phase_3/detailed_feature_scores/{dataset_name}_{model_name}_{method}_scores.csv", index=False)

    # Save the final aggregated report
    os.makedirs("../results/phase_3/final_reports", exist_ok=True)
    results_df = pd.DataFrame(final_results)
    results_df.to_csv(f"../results/phase_3/final_reports/final_results_{dataset_name.lower()}_{model_name.lower()}.csv", index=False)
    

#Mendeley
os.makedirs("../results/phase_3/final_reports", exist_ok=True)
save_stability_redundancy("../results/phase_3/stability_reports/XGBoost_Mendeley_stability_report.csv", 'XGBoost', xgb_mendeley, 'Mendeley', mendeley_sets)
save_stability_redundancy("../results/phase_3/stability_reports/Random Forest_Mendeley_stability_report.csv", 'RF', rf_mendeley, 'Mendeley', mendeley_sets)
save_stability_redundancy("../results/phase_3/stability_reports/Logistic Regression_Mendeley_stability_report.csv", 'LogReg', logreg_mendeley, 'Mendeley', mendeley_sets)

#Kaggle
save_stability_redundancy("../results/phase_3/stability_reports/XGBoost_Kaggle_stability_report.csv", 'XGBoost', xgb_kaggle, 'Kaggle', kaggle_sets)
save_stability_redundancy("../results/phase_3/stability_reports/Random Forest_Kaggle_stability_report.csv", 'RF', rf_kaggle, 'Kaggle', kaggle_sets)
save_stability_redundancy("../results/phase_3/stability_reports/Logistic Regression_Kaggle_stability_report.csv", 'LogReg', logreg_kaggle, 'Kaggle', kaggle_sets)


Evaluating Composite Score for XGBoost - anova
Selected 20 out of 30 features.

Evaluating Composite Score for XGBoost - chi2
Selected 15 out of 25 features.

Evaluating Composite Score for XGBoost - mi
Selected 14 out of 24 features.

Evaluating Composite Score for XGBoost - variance
Selected 17 out of 27 features.

Evaluating Composite Score for RF - anova
Selected 15 out of 25 features.

Evaluating Composite Score for RF - chi2
Selected 12 out of 20 features.

Evaluating Composite Score for RF - mi
Selected 14 out of 25 features.

Evaluating Composite Score for RF - variance
Selected 17 out of 27 features.

Evaluating Composite Score for LogReg - anova
Selected 12 out of 20 features.
{'C': 10, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 5000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'saga', 'tol': 0.001, 'verbose': 0, 'warm_start': False}

Evaluating Compos

C:\Users\nive0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


Selected 26253 out of 29058 features.

Evaluating Composite Score for XGBoost - mi
Selected 21440 out of 24016 features.

Evaluating Composite Score for XGBoost - variance
Selected 25 out of 50 features.

Evaluating Composite Score for RF - anova
Selected 19558 out of 22083 features.

Evaluating Composite Score for RF - chi2


C:\Users\nive0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


Selected 19639 out of 22206 features.

Evaluating Composite Score for RF - mi
Selected 26913 out of 29490 features.

Evaluating Composite Score for RF - variance
Selected 31 out of 57 features.

Evaluating Composite Score for LogReg - anova
Selected 19558 out of 22083 features.
{'C': 10, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 5000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'saga', 'tol': 0.001, 'verbose': 0, 'warm_start': False}

Evaluating Composite Score for LogReg - chi2


C:\Users\nive0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


Selected 26253 out of 29058 features.
{'C': 10, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 5000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'saga', 'tol': 0.001, 'verbose': 0, 'warm_start': False}

Evaluating Composite Score for LogReg - mi
Selected 21440 out of 24016 features.
{'C': 10, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 5000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'saga', 'tol': 0.001, 'verbose': 0, 'warm_start': False}

Evaluating Composite Score for LogReg - variance
Selected 31 out of 57 features.
{'C': 10, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 5000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 